# Home Credit Default Risk — LR vs Random Forest vs XGBoost

This notebook benchmarks three classical/boosted classifiers on the
[Home Credit Default Risk dataset](https://www.kaggle.com/c/home-credit-default-risk)
for the binary task of predicting whether a client defaults on a loan.

For each algorithm we report performance with **default hyperparameters** and
with **hyperparameters tuned via Optuna**.

| Model | Library | GPU |
| :--- | :--- | :--- |
| Logistic Regression | scikit-learn | No |
| Random Forest | scikit-learn | No |
| XGBoost | xgboost | Yes (tuning + final) |

> Companion notebook for the Lending Club dataset uses the same structure.

> **Scope.** FT-Transformer experiments are kept in a separate notebook and are
> not included here.

> **Input data.** This notebook expects a single CSV (`home_credit_default.csv`)
> containing the joined application + supplementary aggregates. The aggregation
> step (mean/count/min/max over `bureau`, `previous_application`,
> `POS_CASH_balance`, `credit_card_balance`, `installments_payments`,
> `bureau_balance`) is performed in a separate ETL notebook upstream of this
> one.

## 1. Setup

In [ ]:
# Install dependencies (uncomment on a fresh environment)
# !pip install -q optuna xgboost shap scikit-learn pandas numpy joblib

In [ ]:
import os
import time
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)
from scipy.stats import ks_2samp

import optuna
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


seed_everything()

### 1.1 Paths

Update `DATA_PATH` to wherever the raw CSV lives on your machine, and
`ARTIFACTS_DIR` to where you want trained models, performance CSVs, and
SHAP tables written.

If you're running on Google Colab, uncomment the Drive-mount cell below
and point both paths into your Drive (e.g.
`/content/drive/MyDrive/credit_risk_modeling/data/...`).

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to
# read data from / write artifacts to your Drive. Skip on local, server,
# or Kaggle runs. The next cell (DATA_PATH / ARTIFACTS_DIR) automatically
# routes through DRIVE_ROOT when it's defined.

# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/credit_risk_modeling"


In [ ]:
# Use Drive paths if the Colab cell above defined DRIVE_ROOT; otherwise local.
# When running locally, repo root is one directory up (notebook is in notebooks/).
# When running on Colab with the cell above uncommented, DRIVE_ROOT takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH = f"{_BASE}/data/home_credit_default.csv"
ARTIFACTS_DIR = f"{_BASE}/artifacts/home_credit"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")


## 2. Domain Cleaning

`preprocess_home_credit` applies the dataset-specific fixes recommended by the
competition discussions:

* Drop rows where `CODE_GENDER == "XNA"`.
* Replace the sentinel value `365243` in every `DAYS_*` column with `NaN`.
* Flip `DAYS_BIRTH` from negative-days-before-application to a positive
  age-in-days representation.
* Coerce every `FLAG*` column to a clean binary integer (0/1).

In [ ]:
def preprocess_home_credit(df):
    """Apply Home-Credit-specific data integrity fixes."""
    df = df.copy()

    # Drop rows with unrecognized gender
    df = df[df["CODE_GENDER"] != "XNA"]

    # Replace sentinel 365243 with NaN in all DAYS_* columns
    days_cols = [c for c in df.columns if c.startswith("DAYS_")]
    for col in days_cols:
        df[col] = df[col].replace(365243, np.nan)

    # DAYS_BIRTH is stored as negative; flip to positive
    if "DAYS_BIRTH" in df.columns:
        df["DAYS_BIRTH"] = -df["DAYS_BIRTH"]

    # Coerce FLAG* columns to clean binary integers
    flag_cols = [c for c in df.columns if c.startswith("FLAG")]
    for col in flag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).clip(0, 1)

    num_cols = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if "TARGET" in num_cols:
        num_cols.remove("TARGET")

    return df, num_cols, cat_cols

## 3. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")

# SK_ID_CURR is just a join key from upstream aggregation; not a feature
if "SK_ID_CURR" in df.columns:
    df = df.drop(columns=["SK_ID_CURR"])

df, num_cols, cat_cols = preprocess_home_credit(df)
print(f"After domain cleaning: {df.shape}")

In [ ]:
print("Dataset overview")
print("-" * 50)
print(f"Total rows   : {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]:,}")
print(f"Numeric cols : {len(num_cols)}")
print(f"Categorical  : {len(cat_cols)}")
print(f"Binary FLAG  : {sum(1 for c in df.columns if 'FLAG' in c)}")
print()
print("Target distribution:")
print((df["TARGET"].value_counts(normalize=True) * 100).round(2).rename("pct"))
print()
print("Top-15 columns by missing rate:")
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
print(missing.head(15).round(2).to_frame("missing_pct"))

## 4. Leakage-Free Preprocessing Pipeline

The pipeline below performs train/validation/test splitting and **fits all
imputers, encoders, and scalers on the training set only**. The same
transformations are then applied to validation and test partitions.

| Step | Description |
| :--- | :--- |
| 1. Stratified split | 64% train / 16% valid / 20% test (stratified on target) |
| 2. Missing-value filter | Drop columns whose train missing-rate > `missing_threshold` |
| 3. Correlation drop | Among numeric features with $\|\rho\| > 0.9$, keep the one with stronger target correlation |
| 4. Numeric imputation | Median (fit on train) |
| 5. Categorical encoding | Fill `"Missing"` then `LabelEncoder` (fit on train, extended to unseen valid/test categories) |
| 6. Numeric scaling | `StandardScaler` (fit on train) |

In [ ]:
def apply_missing_value_filter(X_train, X_valid, X_test, threshold=0.50):
    """Drop columns whose missing rate in X_train exceeds `threshold`."""
    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

    print(f"  Threshold: {threshold:.0%} | Dropped {len(cols_to_drop)} columns")

    X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_valid = X_valid.drop(columns=cols_to_drop, errors="ignore")
    X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]
    return X_train, X_valid, X_test, num_cols, cat_cols


def apply_correlation_drop(X_train, y_train, X_valid, X_test, threshold=0.9):
    """Drop highly correlated numeric features, keeping the one most correlated with the target."""
    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]

    X_train_num = X_train[num_cols]
    corr_matrix = X_train_num.corr().abs()
    target_corr = X_train_num.apply(lambda s: s.corr(y_train)).abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                if row in to_drop or col in to_drop:
                    continue
                weaker = row if target_corr.get(row, 0) < target_corr.get(col, 0) else col
                to_drop.add(weaker)

    print(f"  Numeric features before: {len(num_cols)} | Dropped: {len(to_drop)}")

    X_train = X_train.drop(columns=list(to_drop), errors="ignore")
    X_valid = X_valid.drop(columns=list(to_drop), errors="ignore")
    X_test = X_test.drop(columns=list(to_drop), errors="ignore")

    num_cols = [c for c in num_cols if c not in to_drop]
    return X_train, X_valid, X_test, num_cols, cat_cols


def preprocess_data_pipeline(
    df, target, corr_threshold=0.9, missing_threshold=0.50, random_state=42
):
    """Full preprocessing pipeline: split -> filter -> encode/scale."""
    print("Step 1: Stratified split (64/16/20)")
    X = df.drop(columns=[target])
    y = df[target]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=random_state
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=random_state
    )
    print(
        f"  Train={len(X_train):,} | Valid={len(X_valid):,} | Test={len(X_test):,}"
    )

    print("Step 2: Missing-value column filter")
    X_train, X_valid, X_test, num_cols, cat_cols = apply_missing_value_filter(
        X_train, X_valid, X_test, threshold=missing_threshold
    )

    print("Step 3: Correlation drop (numeric)")
    X_train, X_valid, X_test, num_cols, cat_cols = apply_correlation_drop(
        X_train, y_train, X_valid, X_test, threshold=corr_threshold
    )

    print("Step 4: Median imputation (numeric)")
    median_imputers = {}
    for col in num_cols:
        med = X_train[col].median()
        median_imputers[col] = med
        X_train[col] = X_train[col].fillna(med)
        X_valid[col] = X_valid[col].fillna(med)
        X_test[col] = X_test[col].fillna(med)

    print("Step 5: Label encoding (categorical)")
    encoders = {}
    cat_cardinalities = []
    for col in cat_cols:
        for split in (X_train, X_valid, X_test):
            split[col] = split[col].fillna("Missing")

        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))

        # Extend the encoder with any unseen categories from valid/test
        seen = list(le.classes_)
        for split in (X_valid, X_test):
            for cat in split[col].astype(str).unique():
                if cat not in seen:
                    seen.append(cat)
        le.classes_ = np.array(seen)

        X_valid[col] = le.transform(X_valid[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

        encoders[col] = le
        cat_cardinalities.append(int(X_train[col].nunique()))

    print("Step 6: Standard scaling (numeric)")
    scaler = StandardScaler()
    if num_cols:
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_valid[num_cols] = scaler.transform(X_valid[num_cols])
        X_test[num_cols] = scaler.transform(X_test[num_cols])

    print(
        f"\nFinal feature counts: numeric={len(num_cols)} | "
        f"categorical={len(cat_cols)} | total={X_train.shape[1]}"
    )
    print(f"Categorical cardinalities: {cat_cardinalities}")

    return (
        X_train, y_train, X_valid, y_valid, X_test, y_test,
        num_cols, cat_cols, encoders, median_imputers, scaler, cat_cardinalities,
    )

In [ ]:
(
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    final_num_cols, final_cat_cols,
    encoders, imputers, scaler, cat_cardinalities,
) = preprocess_data_pipeline(
    df, target="TARGET", corr_threshold=0.9, missing_threshold=0.9
)

In [ ]:
print("Dataset Split Summary")
print("-" * 40)
print(f"Train : X = {X_train.shape},  y = {y_train.shape}")
print(f"Valid : X = {X_valid.shape},  y = {y_valid.shape}")
print(f"Test  : X = {X_test.shape},   y = {y_test.shape}")
print()
print(f"Total Features      : {X_train.shape[1]}")
print(f"Numeric Features    : {len(final_num_cols)}")
print(f"Categorical Features: {len(final_cat_cols)}")

## 5. Evaluation Metrics

A single helper computes the metrics we'll report for every model on every
split. The same function is used unchanged across all three algorithms.

* **Discrimination**: AUC, Gini, KS, AUCPR
* **Threshold-dependent (top-decile)**: Precision @ 10%, Recall @ 10%
* **Volume**: Total, Total Bad, Bad Rate

In [ ]:
def evaluate_model_metrics(y_true, y_proba, label="Overall"):
    """Compute discrimination + top-decile metrics for a binary classifier."""
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    total = len(y_true)
    total_bad = int(y_true.sum())
    bad_rate = total_bad / total

    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = np.nan
    gini = 2 * auc - 1
    aucpr = average_precision_score(y_true, y_proba)

    proba_bad = y_proba[y_true == 1]
    proba_good = y_proba[y_true == 0]
    if len(proba_bad) > 0 and len(proba_good) > 0:
        ks_stat, _ = ks_2samp(proba_bad, proba_good)
    else:
        ks_stat = np.nan

    # Top-decile cutoff: 90th percentile of scores -> riskiest 10%
    cutoff = np.percentile(y_proba, 90)
    y_pred_top10 = (y_proba >= cutoff).astype(int)
    prec_top10 = precision_score(y_true, y_pred_top10)
    rec_top10 = recall_score(y_true, y_pred_top10)

    return pd.DataFrame(
        {
            "Segment": label,
            "Total": total,
            "Total Bad": total_bad,
            "Bad Rate": f"{bad_rate:.4f}",
            "AUC": auc,
            "Gini": gini,
            "KS Statistic": ks_stat,
            "AUCPR": aucpr,
            "Precision @10%": prec_top10,
            "Recall @10% (Capture Rate)": rec_top10,
        },
        index=["Value"],
    )


def report_performance(model, X_train, y_train, X_valid, y_valid, X_test, y_test):
    """Score a fitted classifier on all three splits and concat results."""
    train_pred = model.predict_proba(X_train)[:, 1]
    val_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]
    return pd.concat([
        evaluate_model_metrics(y_train, train_pred, label="Train"),
        evaluate_model_metrics(y_valid, val_pred, label="Validation"),
        evaluate_model_metrics(y_test, test_pred, label="Test"),
    ])

## 6. Logistic Regression

### 6.1 Default hyperparameters

In [ ]:
logreg_default = LogisticRegression()

start = time.time()
logreg_default.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.2f}s")

perf_lr_default = report_performance(
    logreg_default, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_lr_default

### 6.2 Hyperparameter tuning with Optuna

In [ ]:
def tune_lr_objective(trial):
    C = trial.suggest_float("C", 1e-5, 100, log=True)
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    use_balanced = trial.suggest_categorical("use_balanced", [True, False])
    class_weight = "balanced" if use_balanced else None

    model = LogisticRegression(
        C=C,
        penalty=penalty,
        solver="liblinear",
        class_weight=class_weight,
        random_state=RANDOM_SEED,
        max_iter=1000,
    )
    model.fit(X_train, y_train)
    val_preds = model.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, val_preds)


N_TRIALS_LR = 20

study_lr = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name="LR_HomeCredit_Tuning"
)

start = time.time()
study_lr.optimize(tune_lr_objective, n_trials=N_TRIALS_LR, show_progress_bar=True)
duration = time.time() - start

print(f"\nTuning complete in {duration / 60:.2f} minutes")
print(f"Best validation AUC: {study_lr.best_value:.5f}")
print("Best params:")
for k, v in study_lr.best_params.items():
    print(f"  {k}: {v}")

### 6.3 Final logistic regression with tuned hyperparameters

In [ ]:
best_params_lr = study_lr.best_params

logreg = LogisticRegression(
    C=best_params_lr["C"],
    penalty=best_params_lr["penalty"],
    class_weight="balanced" if best_params_lr.get("use_balanced") else None,
    solver="liblinear",
    random_state=RANDOM_SEED,
    max_iter=1000,
)

start = time.time()
logreg.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.2f}s")

perf_lr_tuned = report_performance(
    logreg, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_lr_tuned

In [ ]:
perf_lr_tuned.to_csv(os.path.join(ARTIFACTS_DIR, "lr_perf.csv"))
joblib.dump(logreg, os.path.join(ARTIFACTS_DIR, "logistic_regression.pkl"))

## 7. Random Forest

### 7.1 Default hyperparameters

In [ ]:
rf_default = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)

start = time.time()
rf_default.fit(X_train, y_train)
print(f"Training time: {(time.time() - start) / 60:.2f} minutes")

perf_rf_default = report_performance(
    rf_default, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_rf_default

### 7.2 Hyperparameter tuning with Optuna

In [ ]:
def tune_rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "max_samples": trial.suggest_float("max_samples", 0.6, 0.9),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 100),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 50),
        "max_features": trial.suggest_float("max_features", 0.6, 0.9),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
    }
    model = RandomForestClassifier(
        **params,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    model.fit(X_train, y_train)
    val_pred = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, val_pred)
    print(f"Trial {trial.number} -> AUC: {auc:.5f}")
    return auc


N_TRIALS_RF = 10

study_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name="RF_HomeCredit_Tuning")

start = time.time()
study_rf.optimize(tune_rf_objective, n_trials=N_TRIALS_RF, show_progress_bar=True)
duration = time.time() - start

print(f"\nTuning complete in {duration / 60:.2f} minutes")
print(f"Best validation AUC: {study_rf.best_value:.5f}")
print(f"Best params: {study_rf.best_params}")

### 7.3 Final random forest with tuned hyperparameters

In [ ]:
best_params_rf = study_rf.best_params

rf = RandomForestClassifier(
    **best_params_rf,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

start = time.time()
rf.fit(X_train, y_train)
print(f"Training time: {(time.time() - start) / 60:.2f} minutes")

perf_rf_tuned = report_performance(
    rf, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_rf_tuned

In [ ]:
perf_rf_tuned.to_csv(os.path.join(ARTIFACTS_DIR, "rf_perf.csv"))
joblib.dump(rf, os.path.join(ARTIFACTS_DIR, "random_forest.pkl"))

## 8. XGBoost (GPU)

XGBoost uses GPU acceleration via `device="cuda"` for both hyperparameter
tuning and the final fit. Set `device="cpu"` if you don't have a CUDA GPU
available.

### 8.1 Default hyperparameters

In [ ]:
EARLY_STOPPING_ROUNDS = 50

xgb_default_config = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

model_xgb_default = xgb.XGBClassifier(**xgb_default_config)

start = time.time()
model_xgb_default.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=100,
)
print(f"Training time: {(time.time() - start) / 60:.2f} minutes")

perf_xgb_default = report_performance(
    model_xgb_default, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_xgb_default

### 8.2 Hyperparameter tuning with Optuna

`scale_pos_weight` is searched in a window centered on the empirical
imbalance ratio, $\text{neg}/\text{pos}$, to give Optuna a sensible band
around the natural class balance.

In [ ]:
def tune_xgb_objective(trial):
    neg_count = len(y_train) - y_train.sum()
    pos_count = y_train.sum()
    theoretical_ratio = neg_count / pos_count
    spw_min = theoretical_ratio * 0.5
    spw_max = theoretical_ratio * 2.0

    params = {
        # Fixed config
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda",
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        # Search space
        "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "lambda": trial.suggest_float("lambda", 1.0, 5.0),
        "alpha": trial.suggest_float("alpha", 1.0, 5.0),
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", spw_min, spw_max, log=True
        ),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 50),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    val_preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, val_preds)
    print(f"Trial {trial.number} -> AUC: {auc:.5f}")
    return auc


N_TRIALS_XGB = 30

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name="XGBoost_HomeCredit_Tuning"
)

start = time.time()
study_xgb.optimize(tune_xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)
duration = time.time() - start

print(f"\nTuning complete in {duration / 60:.2f} minutes "
      f"({duration / N_TRIALS_XGB:.2f}s per trial)")
print(f"Best validation AUC: {study_xgb.best_value:.5f}")
print("Best params:")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")

### 8.3 Final XGBoost with tuned hyperparameters

In [ ]:
best_params_xgb = study_xgb.best_params

xgb_final_config = {
    **best_params_xgb,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

model_xgb = xgb.XGBClassifier(**xgb_final_config)

start = time.time()
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=100,
)
print(f"Training time: {(time.time() - start) / 60:.2f} minutes")

perf_xgb_tuned = report_performance(
    model_xgb, X_train, y_train, X_valid, y_valid, X_test, y_test
)
perf_xgb_tuned

In [ ]:
perf_xgb_tuned.to_csv(os.path.join(ARTIFACTS_DIR, "xgb_perf.csv"))
model_xgb.save_model(os.path.join(ARTIFACTS_DIR, "xgb_model.json"))

## 9. SHAP Feature Importance

Tree-based SHAP values for the tuned XGBoost and Random Forest models. The
helpers below produce a global importance table (mean absolute SHAP value per
feature) and the standard bar / beeswarm summary plots.

In [ ]:
import shap
import matplotlib.pyplot as plt


def get_shap_importance_xgb(model, X_data):
    """SHAP global importance for an XGBoost classifier."""
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_data)

    feature_names = (
        X_data.columns
        if isinstance(X_data, pd.DataFrame)
        else [f"Feature {i}" for i in range(shap_values.shape[1])]
    )
    importance_df = (
        pd.DataFrame(
            {
                "Feature": feature_names,
                "SHAP_Value_Contribution": np.abs(shap_values).mean(axis=0),
            }
        )
        .sort_values("SHAP_Value_Contribution", ascending=False)
        .reset_index(drop=True)
    )
    return importance_df, shap_values


def get_shap_importance_rf(model, X_data):
    """SHAP global importance for a sklearn Random Forest classifier.

    Handles both the 'list of arrays' and the newer '3D ndarray' SHAP outputs,
    selecting the positive class.
    """
    explainer = shap.TreeExplainer(model)
    raw = explainer.shap_values(X_data)

    if isinstance(raw, list):
        shap_values = raw[-1]
    elif isinstance(raw, np.ndarray) and raw.ndim == 3:
        shap_values = raw[:, :, 1]
    else:
        shap_values = raw

    importance_df = (
        pd.DataFrame(
            {
                "Feature": X_data.columns,
                "SHAP_Value_Contribution": np.abs(shap_values).mean(axis=0),
            }
        )
        .sort_values("SHAP_Value_Contribution", ascending=False)
        .reset_index(drop=True)
    )
    return importance_df, shap_values


def plot_shap_summary(shap_values, X_data, max_features=15, save_prefix=None):
    """Plot SHAP bar and beeswarm summaries."""
    n = min(max_features, X_data.shape[1])

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_data, plot_type="bar", max_display=n, show=False)
    plt.title("SHAP Global Feature Importance")
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f"{save_prefix}_bar.png", dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_data, max_display=n, show=False)
    plt.title("SHAP Feature Impact and Direction")
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f"{save_prefix}_beeswarm.png", dpi=200, bbox_inches="tight")
    plt.show()

### 9.1 SHAP — XGBoost

In [ ]:
shap_xgb_df, shap_xgb_values = get_shap_importance_xgb(model_xgb, X_test)
shap_xgb_df.head(20)

In [ ]:
shap_xgb_df.to_csv(os.path.join(ARTIFACTS_DIR, "shap_importance_xgb.csv"), index=False)
plot_shap_summary(
    shap_xgb_values,
    X_test,
    max_features=10,
    save_prefix=os.path.join(ARTIFACTS_DIR, "shap_xgb"),
)

### 9.2 SHAP — Random Forest

In [ ]:
shap_rf_df, shap_rf_values = get_shap_importance_rf(rf, X_test)
shap_rf_df.head(20)

In [ ]:
shap_rf_df.to_csv(os.path.join(ARTIFACTS_DIR, "shap_importance_rf.csv"), index=False)
plot_shap_summary(
    shap_rf_values,
    X_test,
    max_features=10,
    save_prefix=os.path.join(ARTIFACTS_DIR, "shap_rf"),
)

## 10. Summary

The table below collects the test-set headline metrics across all six
configurations (default vs tuned for each model).

In [ ]:
def extract_test_row(perf_df, model_name):
    row = perf_df[perf_df["Segment"] == "Test"].iloc[0].copy()
    row["Model"] = model_name
    return row


summary = pd.DataFrame([
    extract_test_row(perf_lr_default, "LogisticRegression (default)"),
    extract_test_row(perf_lr_tuned, "LogisticRegression (tuned)"),
    extract_test_row(perf_rf_default, "RandomForest (default)"),
    extract_test_row(perf_rf_tuned, "RandomForest (tuned)"),
    extract_test_row(perf_xgb_default, "XGBoost (default)"),
    extract_test_row(perf_xgb_tuned, "XGBoost (tuned)"),
])

summary = summary[[
    "Model", "AUC", "Gini", "KS Statistic", "AUCPR",
    "Precision @10%", "Recall @10% (Capture Rate)",
]]
summary = summary.set_index("Model")
summary.to_csv(os.path.join(ARTIFACTS_DIR, "summary_test_metrics.csv"))
summary